# Conditional alphas {#sec-alphas}

Now, we need to get the conditional alphas from the risk-adjusted returns and state variables.

The following math and commentary is taken largely as-is from the paper. First, I present the theory, then the implementation in Python, and finally, the results.

## Methodology

The methodology is taken from @Distaso2013, section 2.2.

In [src/betas.ipynb](src/betas.ipynb)/@sec-betas, we calculated the risk-adjusted returns $\hat{Z}_{i, t+1, M}$. This is useful because $Z_{i,t+1} = \alpha_{i,t} + \epsilon_{i,t+1} + \eta_{i,t+1}$. Identification of the conditional alpha stems from the fact that the conditional expectations of $\epsilon_{i,t+1}$ and $\eta_{i,t+1}$ are zero, so that $\alpha_{i,t} \equiv \mathbb{E}(Z_{i,t+1}|C_t)$.

Thus, it suffices to estimate the conditional expectation of the risk-adjusted returns, $\mathbb{E}(Z_{i,t+1}|C_t)$, to back out conditional alphas.

After estimating $\hat{Z}_{i, t+1, M}$ with the realized betas, we only need to estimate the conditional expectation $\mathbb{E}(\hat{Z}_{i,t+1,M}|C_t)$ of the realized risk-adjusted returns using nonparametric kernel methods. The paper proposes a consistent estimator with respect to the number of intraday observations $M$.

The high dimensionality of the state variable vector $C_t$ clashes with the nonparametric kernel estimator convergence, so we need to summarize it with PCA. Let then $\widehat{PC}_t = (\widehat{PC}_{1,t}, \dots, \widehat{PC}_{k,t})'$ denote the $k \le k_C$ principal component estimates of $C_t$.

The conditional alpha estimator is:

$$
\hat{\alpha}_{i,t} = \hat{m}_{i,T,M}(\widehat{PC}_t) = \frac{\frac{1}{T h_T^k} \sum_{\ell=1}^{T-1} \hat{Z}_{i,\ell+1,M} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_T}\right)}{\hat{g}_T(\widehat{PC}_t)}
$$

where $\hat{g}_T(\widehat{PC}_t) = \frac{1}{T h_T^k} \sum_{\ell=1}^{T-1} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_T}\right)$ is the kernel estimator of the density $g$ of the vector of principal components. 

To circumvent the difficulty of estimating $m_i(pc)$ over regions of low density, we consider a trimmed version of $\hat{m}_{i,T,M}(pc)$ as in Andrews (1995):

$$
tr_T\{\hat{m}_{i,T,M}(pc)\} = \hat{m}_{i,T,M}(pc) 1\{pc \in \hat{G}_T(pc)\}
$$

where $\hat{G}_T(pc) = \{c : \hat{g}_T(pc) > d_T\}$.

<!-- Instead of trimming, one could gauge the magnitude of the conditional alphas relative to the noise level by means of Treynor and Black’s (1973) appraisal ratio, which gauges the extra units of unsystematic risk investors have to bear for deviating from a passive management strategy. We explore this alternative in Appendix B.2. -->

### Expanding window calculation

The paper proposes an expanding window to restrict attention to real-time information: 

> To obtain conditional alphas, we entertain an expanding window to restrict attention to real-time information. We estimate the conditional expectation of the risk-adjusted returns using either an affine specification or a kernel-regression approach using a multivariate standard Gaussian kernel with rule-of-thumb bandwidths. As for the amount of trimming in (10), uniform consistency requires $d_T = O(h_T^\zeta)$ with $0 < \zeta < 1/4$. Note that, because $h_T \to 0$ as $T \to \infty$, trimming becomes more aggressive as $\zeta$ declines. We set $\zeta$ to 0.20 to enforce only a mild trimming, though we also experiment with other values of $\zeta$ in Section 4.2.

> The daily frequency of the analysis dictates the choice of state variables. In particular, we consider changes in the VIX index and in the Fed Fund rate, credit and term spreads, momentum, size and value factors, short- and long-term reversal factors, and volatility risk premium (namely, difference between the VIX index and realized volatility). To avoid the curse of dimensionality, we consider only their three first principal components.

This windowed version calculates $\widehat{PC}_t$ using data from $1:t$ only. And the bandwidth $h_T$ and trimming threshold $d_T$ are calculated separately for each $t$ using $t$ as $T$. Additionally, the kernel estimator does not uses data up to $T$, but only up to $t-1$. To differentiate the notation, let $W = t$ be the total window size:

$$
\hat{\alpha}_{i,t} = \hat{m}_{i,W,M}(\widehat{PC}_t) = \frac{\frac{1}{W h_W^k} \sum_{\ell=1}^{W-1} \hat{Z}_{i,\ell+1,M} K\left(\frac{\widehat{PC}_\ell - \widehat{PC}_t}{h_W}\right)}{\hat{g}_W(\widehat{PC}_t)}
$$

<!-- CHECK: should it be fixed h_T and d_T still? -->

## Code setup

Core modules:

In [118]:
#| output: false
import os
if os.path.basename(os.getcwd()) == "src": os.chdir("..")

import numpy as np
import polars as pl

from scipy.io import loadmat as _loadmat
from tqdm import tqdm

from typing import NamedTuple, Final
from numpy.typing import NDArray

from src.parameters import PARAMETERS as PARS

class sio:
    loadmat = staticmethod(_loadmat)

Parameters:

In [119]:
class AlphasParameters(NamedTuple):
    window_warmup: int # Initial burn-in window size
    scaler: float      # Bandwidth scaling factor
    trim_exp: float    # Trimming exponent zeta in (0, 1/4)
    k: int         # Number of principal components to retain


PARS_ALPHAS: Final[AlphasParameters] = AlphasParameters(
    window_warmup = 500,
    scaler = 1.0,
    trim_exp = 0.2,
    k = 3
)

DATE_START, DATE_END = (
    pl.lit(x.astype("datetime64[D]").astype(np.uint32))
    for x in [PARS.date_start, PARS.date_end]
)

State variables loading:

In [120]:
state_mat = sio.loadmat("data/states_raw/FactorData_2018.mat")
print(f"state_mat's keys: {list(state_mat.keys())[3:]}")

state_raws = []
for key in ["factors", "ADSdata"]:
    mat = state_mat[key]
    data = (
        pl.DataFrame(mat[:, 1:])
        .with_columns(ts_day_ny = (mat[:, 0] - 719_529).astype(np.uint32))
        .select(pl.col("ts_day_ny"), pl.all().exclude("ts_day_ny"))
    )
    state_raws.append(data)

data_state = (
    state_raws[0]
    .join(state_raws[1], on = "ts_day_ny", how = "inner")
    .filter(
        (pl.col("ts_day_ny") >= DATE_START)
        & (pl.col("ts_day_ny") <= DATE_END)
    )
    .with_columns(pl.all().fill_nan(None))
    .pipe(lambda df:
        df.select(
            col for col in df.columns
            if not (df[col].is_null().any() or df[col].is_nan().any())
        )
    )
    .tail(-1) # To match with adjusted returns' dropped 1st observation
)

data_state

state_mat's keys: ['ADSdata', 'ADSvin', 'facDates', 'facHead', 'factors', 'ffdata']


ts_day_ny,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_830,column_831,column_832,column_833,column_834,column_835,column_836,column_837,column_838,column_839,column_840,column_841,column_842,column_843,column_844,column_845,column_846,column_847,column_848,column_849,column_850,column_851,column_852,…,column_860,column_861,column_862,column_863,column_864,column_865,column_866,column_867,column_868,column_869,column_870,column_871,column_872,column_873,column_874,column_875,column_876,column_877,column_878,column_879,column_880,column_881,column_882,column_883,column_884,column_885,column_886,column_887,column_888,column_889,column_890,column_891,column_892,column_893,column_894,column_895,column_896
u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
11325,-3.39,-0.32,-5.04,6.99,-7.27,-0.67,-4.15,-2.53,-5.93,0.86,-0.55,-23.663841,50.263841,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.76065,-0.762034,-0.762034,-0.762034,-0.762034,-0.762034,-0.762034,-0.762034,…,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.754846,-0.750949,-0.751206,-0.751206,-0.751206,-0.751206,-0.751206,-0.751206,-0.751206,-0.790969,-0.790969,-0.790969,-0.790969,-0.790969,-0.790969,-0.790969,-0.790969
11326,0.37,-0.43,-0.62,-0.42,-4.13,0.83,0.22,0.28,-0.79,0.85,-0.34,6.952818,20.017182,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.777482,-0.77871,-0.77871,-0.77871,-0.77871,-0.77871,-0.77871,-0.77871,…,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.771355,-0.767683,-0.768464,-0.768464,-0.768464,-0.768464,-0.768464,-0.768464,-0.768464,-0.808035,-0.808035,-0.808035,-0.808035,-0.808035,-0.808035,-0.808035,-0.808035
11327,1.7,-0.09,1.9,-0.35,0.66,0.74,2.19,1.92,2.32,0.84,-0.19,3.863827,24.806173,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.79431,-0.795388,-0.795388,-0.795388,-0.795388,-0.795388,-0.795388,-0.795388,…,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.787878,-0.784423,-0.785709,-0.785709,-0.785709,-0.785709,-0.785709,-0.785709,-0.785709,-0.825037,-0.825037,-0.825037,-0.825037,-0.825037,-0.825037,-0.825037,-0.825037
11330,1.17,0.15,1.49,-0.89,0.15,-0.36,0.96,0.98,0.66,0.81,-0.25,15.421262,14.418738,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.8441,-0.84475,-0.84475,-0.84475,-0.84475,-0.84475,-0.84475,-0.84475,…,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.836856,-0.834003,-0.836692,-0.836692,-0.836692,-0.836692,-0.836692,-0.836692,-0.836692,-0.875067,-0.875067,-0.875067,-0.875067,-0.875067,-0.875067,-0.875067,-0.875067
11331,-1.85,-0.03,0.44,1.02,-1.56,0.23,-0.24,-0.92,-0.51,0.82,-0.26,8.027593,19.962407,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-0.860389,-

Returns loading:

In [121]:
data_adjusted = pl.scan_parquet("data/stocks_1min_adjusted.parquet")

Checking data alignment:

In [122]:
range_states = tuple(f(data_state["ts_day_ny"]) for f in [min, max])

range_returns = (
    data_adjusted
    .select(
        min = pl.col("ts_day_ny").min(),
        max = pl.col("ts_day_ny").max()
    )
    .collect()
    .row(0)
)

if range_states != range_returns:
    raise ValueError(
        f"State and returns data have different date ranges: "
        f"{range_states} vs {range_returns}"
    )

## Calculating window PCs

Let $W$ denote the total size of a window, which is equal to $t$.

In [123]:
def pca_window(
    C_1_W: NDArray[np.float64], # (W, k) historical state variables
    k: int
) -> NDArray[np.float64]:
    # Standardization:
    mean = np.mean(C_1_W, axis = 0) # (k,) Across t
    std = np.std(C_1_W, axis = 0, ddof = 1) # (k,) Across t
    if np.any(std == 0.0):
        raise ValueError("Constant feature detected in PCA window.")

    C_1_W_norm = (C_1_W - mean) / std

    # Eigendecomposition:
    cov_mat = np.cov(C_1_W_norm, rowvar = False, ddof = 1) # (k, k)
    eigvals, eigvecs = np.linalg.eigh(cov_mat)

    # Principal components:
    idx_sort = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, idx_sort]
    scores = C_1_W_norm @ eigvecs[:, :k] # (T, k)

    # Rescaling:
    scores = scores / 100.0
    score_mean = np.mean(scores, axis = 0)
    score_std = np.std(scores, axis = 0, ddof = 1)
    if np.any(score_std == 0.0):
        raise ValueError("Constant feature detected in PCs window.")

    return (scores - score_mean) / score_std

# CHECK: if division by zero exist, should we just std[std == 0.0] = 1.0?

In [ ]:
class WindowArgs(NamedTuple):
    t: int
    idx_1_Wm1: NDArray[np.uint32] # (W,) indices of the window
    PC_1_Wm1: NDArray[np.float64]  # (W-1, k) historical PC predictors
    PC_t: NDArray[np.float64]      # (k,) current PC predictor at day t
    h_W: np.float64
    d_W: np.float64

def get_windows_data(
    data_state: pl.DataFrame
) -> list[WindowArgs]:
    windows_args = []
    ts = data_state["ts_day_ny"][(PARS_ALPHAS.window_warmup - 1):-1]

    for t in tqdm(ts):
        data_state_w = (
            data_state
            .filter(pl.col("ts_day_ny") <= t)
        )

        idx_1_Wm1 = (
            data_state_w["ts_day_ny"].to_numpy()[:-1]
        )
        window_data = (
            data_state_w.select(pl.all().exclude("ts_day_ny")).to_numpy()
        )

        # Principal components:
        W, k = window_data.shape[0], PARS_ALPHAS.k # CHECK: eq to scores.shape
        scores = pca_window(window_data, k)
        PC_1_Wm1 = scores[:-1, :] # Historical PC predictors
        PC_t = scores[-1, :]  # Current PC predictor at day t

        # Kernel bandwidth:
        bandrate = (
            (4.0 / (k + 2.0)) ** (1.0 / (k + 4.0))
            * (W**(-1.0 / (k + 4.0)))
        )
        smx = np.sqrt(np.mean(np.std(scores, axis = 0, ddof = 1)**2))
        h_W = PARS_ALPHAS.scaler * bandrate * smx

        # Density trimming threshold:
        d_W = (bandrate * smx) ** PARS_ALPHAS.trim_exp

        windows_args.append(WindowArgs(t, idx_1_Wm1, PC_1_Wm1, PC_t, h_W, d_W))

    return windows_args

In [143]:
windows_args = get_windows_data(data_state)

100%|██████████| 3733/3733 [00:41<00:00, 89.38it/s] 


## Calculating conditional alphas

In [126]:
def kernel_gauss(x: NDArray[np.float64]) -> NDArray[np.float64]:
    return (1.0 / np.sqrt(2.0 * np.pi)) * np.exp(-0.5 * x**2)

def cae_gauss_trim(
    PC_t: NDArray[np.float64],     # (k,) conditioning state variables at time t (PC_t)
    Z_2_W: NDArray[np.float64],    # (W-1,) target risk-adjusted returns (Z_{2:W})
    PC_1_Wm1: NDArray[np.float64], # (W-1, k) historical state variables (PC_{1:W-1})
    h_W: NDArray[np.float64],      # (k,) repeating kernel bandwidth (h_W)
    d_W: np.float64                # Trimming density threshold (d_W)
) -> float:
    K_1_Wm1 = (
        (1.0 / np.prod(h_W))
        * np.prod(kernel_gauss((PC_1_Wm1 - PC_t) / h_W), axis = 1) # Across k
    ) # (T-1,)
    deno = np.mean(K_1_Wm1) # Across t

    if deno <= d_W or np.isnan(deno):
        return 0.0

    num = np.mean(K_1_Wm1 * Z_2_W) # Across t
    return float(num / deno)
# CHECK: can we change the repeating h_W vector into a scalar h_W?

def cae_linear(
    PC_t: NDArray[np.float64], # (k,)
    Z_2_W: NDArray[np.float64],    # (T-1,)
    PC_1_Wm1: NDArray[np.float64]    # (T-1, k)
) -> np.float64:
    X = np.column_stack([np.ones(PC_1_Wm1.shape[0]), PC_1_Wm1])

    beta, _, _, _ = np.linalg.lstsq(X, Z_2_W)
    return np.append(1.0, PC_t) @ beta

In [ ]:
permnos = pl.read_csv("data/stocks_raw/counts_permno.csv")['permno'].unique()
ts = data_state["ts_day_ny"][(PARS_ALPHAS.window_warmup - 1):-1]
results = []

for permno in tqdm(permnos):
        data_permno_2_T = (
            data_adjusted
            .filter(pl.col("permno") == permno).collect()
            .tail(-1)
        )

        permno_t_min = data_permno_2_T["ts_day_ny"].min()
        first_obs = (ts >= permno_t_min).arg_true()[0]
        # TODO: probably unneeded after liquidity filter

        for w_args in windows_args[first_obs+50:]:
            t, idx_1_Wm1, PC_1_Wm1, PC_t, h_W, d_W = w_args

            data_join = (
                pl.DataFrame({
                    "ts_day_ny": idx_1_Wm1,
                    "PC_1_Wm1": PC_1_Wm1
                })
                .join(
                    data_permno_2_T, on = "ts_day_ny", how = "inner"
                )
            )

            PC_1_Wm1 = data_join["PC_1_Wm1"].to_numpy()
            Z_2_W = data_join["adjusted_return"].to_numpy()

            print(PC_1_Wm1.shape, Z_2_W.shape, t)

            # Nonparametric Kernel Alpha Estimate:
            alpha_kernel = cae_gauss_trim(
                PC_t = PC_t,
                Z_2_W = Z_2_W,
                PC_1_Wm1 = PC_1_Wm1,
                h_W = h_W * np.ones_like(PC_t),
                d_W = d_W
            )

            # Linear Alpha Benchmark Estimate:
            alpha_linear = cae_linear(
                PC_t = PC_t,
                Z_2_W = Z_2_W,
                PC_1_Wm1 = PC_1_Wm1
            )

            results.append({
                'permno': permno,
                'ts_day_ny': t,
                'alpha': alpha_kernel,
                'linear_alpha': alpha_linear
            })